# Read one node's view metrics: EMA spread

Hypothesis: a positive fast/slow EMA spread predicts positive subsequent returns. This is a workflow example, not an established edge. See `examples/EMA_RESEARCH.md`.

The node is replayed only by a backtest: `sugarloaf run --adapter quantconnect --manifest deployments/quantconnect/ema_forecasts.toml` runs the score alone as a forecast run. At the end of the backtest the launcher scores each view against what SPY then did and writes the metrics to the backtest log (`VIEW_METRICS`). This notebook runs locally, with `QC_USER_ID` and `QC_API_TOKEN` set: it reads a finished run's log through the REST API, saves the output on disk by backtest id, and shows the metrics.

In [ ]:
from pathlib import Path

from sugarloaf.adapters.quantconnect.cloud.backtests import read_run_output, save_run_output
from sugarloaf.adapters.quantconnect.cloud.client import default_client

PROJECT = 37377184
# The backtest id `sugarloaf run` prints.
BACKTEST = ""
assert BACKTEST, "Set BACKTEST to the id of a finished ema_forecasts backtest"
output = read_run_output(default_client(), PROJECT, BACKTEST)
path = save_run_output(output, Path("artifacts/ema_forecasts") / f"{BACKTEST}-output.json")
print("Saved:", path, "errors:", output["errors"], output["summary"]["errors"])
output["metrics"]["counts"]

The score is raw, so its metrics are directional: the hit rate compares the sign of the score with the sign of the realized return; precision, recall and F1 treat an up move as the positive class. The deciles group views by score and give the realized return's mean, median and quantiles per group; `path` is the group's mean return at ten evenly spaced times from signal time to the target's end.

In [ ]:
score = output["metrics"]["kinds"]["score"]["pooled"]
print({k: score[k] for k in ("count", "hit_rate", "precision", "recall", "f1", "decile_spread")})
for group in score["deciles"]:
    print(group["group"], group["count"], group["key_min"], group["key_max"],
          group["mean"], group["q0.1"], group["q0.9"], group["path"])

Record the source, configuration, dates, results, failed variants and limitations beside the node. The notebook ships without generated outputs.